# CSV

We will do the following:
- download the CSV from Hugging Face
- convert the CSV to parquet
- upload the parquet to Hugging Face


## Setup


In [ ]:
%%capture pkg_output
%%bash
apt-get update
apt-get install -y tree jq


In [ ]:
import requests
import pandas
import pandas as pd
import duckdb
from google.colab import userdata
import os


## Converting a remote CSV file to parquet



In [ ]:
url = 'https://huggingface.co/datasets/rwcitek/DUC-2026/resolve/main/sub-est2025.csv'
base_file = url.split("/")[-1].split(".")[0]

url, base_file


('https://huggingface.co/datasets/rwcitek/DUC-2026/resolve/main/sub-est2025.csv',
 'sub-est2025')

### Extract, transform, load ( ETL ) in one step


In [ ]:
# Connect to an in-memory database
con = duckdb.connect()

# Read the remote CSV and write it directly to a local Parquet file
con.execute(f"""
    COPY (
        SELECT *
        FROM read_csv("{url}", encoding = 'latin-1', all_varchar = true)
    )
    TO '{base_file}.parquet'
    (FORMAT PARQUET);
""")

! ls -l {base_file}.parquet


-rw-r--r-- 1 root root 3406654 Nov 18 16:20 sub-est2025.parquet


## Verify


In [ ]:
duckdb.query(f'''
  select *
  from "{base_file}.parquet"
  limit 10
  ;
''').df()


,SUMLEV,STATE,COUNTY,PLACE,COUSUB,CONCIT,PRIMGEO_FLAG,FUNCSTAT,NAME,STNAME,ESTIMATESBASE2020,POPESTIMATE2020,POPESTIMATE2021,POPESTIMATE2022,POPESTIMATE2023,POPESTIMATE2024,POPESTIMATE2025
0,040,01,000,00000,00000,00000,0,A,Alabama,Alabama,5025437,5032962,5050058,5076868,5117850,5163055,5193088
1,162,01,000,00124,00000,00000,0,A,Abbeville city,Alabama,2355,2355,2340,2338,2351,2391,2378
2,162,01,000,00460,00000,00000,0,A,Adamsville city,Alabama,4406,4394,4329,4264,4219,4208,4178
3,162,01,000,00484,00000,00000,0,A,Addison town,Alabama,661,659,665,669,666,672,669
4,162,01,000,00676,00000,00000,0,A,Akron town,Alabama,229,228,228,226,231,234,236
5,162,01,000,00820,00000,00000,0,A,Alabaster city,Alabama,33341,33398,33732,33890,34177,34463,34748
6,162,01,000,00988,00000,00000,0,A,Albertville city,Alabama,22390,22407,22524,22788,23159,23522,23710
7,162,01,000,01132,00000,00000,0,A,Alexander City city,Alabama,14856,14833,14700,14615,14503,14498,14515
8,162,01,000,01228,00000,00000,0,A,Aliceville city,Alabama,2177,2169,2130,2082,2075,2044,2024
9,162,01,000,01396,00000,00000,0,A,Allgood town,Alabama,540,539,542,547,550,557,557


In [ ]:
duckdb.query(f'''
  describe
  select *
  from "{base_file}.parquet"
  limit 10
  ;
''').df()


,column_name,column_type,null,key,default,extra
0,SUMLEV,VARCHAR,YES,None,None,None
1,STATE,VARCHAR,YES,None,None,None
2,COUNTY,VARCHAR,YES,None,None,None
3,PLACE,VARCHAR,YES,None,None,None
4,COUSUB,VARCHAR,YES,None,None,None
5,CONCIT,VARCHAR,YES,None,None,None
6,PRIMGEO_FLAG,VARCHAR,YES,None,None,None
7,FUNCSTAT,VARCHAR,YES,None,None,None
8,NAME,VARCHAR,YES,None,None,None
9,STNAME,VARCHAR,YES,None,None,None


#### Milestone

We downloaded a CSV file from Hugging Face, converted it to parquet, and saved it on the Colab VM.

We now have a local copy of the parquet file that we can work with.


## Push to Hugging Face

### Configure settings and authenticate to Hugging Face


In [ ]:
hf_account = os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account


'rwcitek'

In [ ]:
hf_repo = os.environ["HF_REPO"] = "DUC-2026"
hf_repo


'DUC-2026'

In [ ]:
hf_file = os.environ["HF_FILE"] = f"{base_file}.parquet"
hf_file


'sub-est2025.parquet'

In [ ]:
# Retrieve the token from Colab's secret manager
_ = hf_token = os.environ["HF_TOKEN"] = userdata.get('hf_token')
f"{_[:3]} ... {_[-4:]}"


'hf_ ... dQPb'

In [ ]:
# Log in automatically without an interactive prompt
! hf auth login --token ${HF_TOKEN}


Hint: A new version of huggingface_hub (2.4.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Colab` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Push the parquet file to Hugging Face


In [ ]:
%%capture hf_upload
%%bash

TARGET_REPO="${HF_ACCOUNT}/${HF_REPO}"
LOCAL_SOURCE="./${HF_FILE}"
REMOTE_DESTINATION="./${HF_FILE}"

# Comment out the echo to make this live
# echo DRY-RUN: \
hf upload \
  --type dataset \
  "${TARGET_REPO}" \
  "${LOCAL_SOURCE}" \
  "${REMOTE_DESTINATION}"


In [ ]:
print(hf_upload.stdout)


✓ Uploaded
  url: https://huggingface.co/datasets/rwcitek/DUC-2026/commit/ec1273a809236b5e8649cacd1cbd7344cabf5ac2



## Query the parquet on Hugging Face using DuckDB


In [ ]:
# Construct the Hugging Face URL for your dataset file
hf_url = f"https://huggingface.co/datasets/{hf_account}/{hf_repo}/resolve/main/{hf_file}"
hf_url


'https://huggingface.co/datasets/rwcitek/DUC-2026/resolve/main/sub-est2025.parquet'

In [ ]:
# Query the remote parquet
query = (f'''
  select count(1) as count
  from "{hf_url}"
  ;
''')
print(query)
duckdb.sql(query).df()



  select count(1) as count
  from "https://huggingface.co/datasets/rwcitek/DUC-2026/resolve/main/sub-est2025.parquet"
  ;



,count
0,81353


In [ ]:
# Query the remote parquet
query = (f'''
  describe
  select *
  from "{hf_url}"
  ;
''')
print(query)
duckdb.sql(query).df()



  describe
  select *
  from "https://huggingface.co/datasets/rwcitek/DUC-2026/resolve/main/sub-est2025.parquet"
  ;



,column_name,column_type,null,key,default,extra
0,SUMLEV,VARCHAR,YES,None,None,None
1,STATE,VARCHAR,YES,None,None,None
2,COUNTY,VARCHAR,YES,None,None,None
3,PLACE,VARCHAR,YES,None,None,None
4,COUSUB,VARCHAR,YES,None,None,None
5,CONCIT,VARCHAR,YES,None,None,None
6,PRIMGEO_FLAG,VARCHAR,YES,None,None,None
7,FUNCSTAT,VARCHAR,YES,None,None,None
8,NAME,VARCHAR,YES,None,None,None
9,STNAME,VARCHAR,YES,None,None,None


## Summary

We did the following:
- downloaded the CSV file from Hugging Face
- converted the CSV file to Parquet, with some options
- queried the parquet file
- pushed the local parquet file to Hugging Face
- queried the remote parquet file on Hugging Face

